In [ ]:
#%pip install python-dotenv

In [ ]:
#%pip install openai

In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5-nano",
    input="Write a one-sentence bedtime story about a unicorn."
)

print(response.output_text)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import numpy as np

In [ ]:
from openai import OpenAI
# client = OpenAI(api_key = os.environ['OPENAI_API_KEY'])
client = OpenAI()
response = client.responses.create(
    model='gpt-5-nano',
    input='이번 2025년도 크리스마스에, 잠실에 열리는 크리스마스 이벤트에 대해서 설명해줘'
)

print(response.output_text)

In [ ]:
from openai import OpenAI
# client = OpenAI(api_key = os.environ['OPENAI_API_KEY'])
client = OpenAI()
def ask_llm(prompt, model='gpt-5-nano',temp=1):
    response = client.chat.completions.create(
        model=model,
        temperature=temp,
        messages = [
            {"role": "user", "content": prompt}
                    ]
    )
    return response.choices[0].message.content
    
# temperature / temp
# temperature는 창의성(무작위성)을 조절하는 파라미터야.
# 의미 :
# 0에 가까울수록: 정답이 고정됨, 가장 논리적·일관된 답을 선택
# 높을수록: 다양하고 창의적인 답변을 생성


# completions.create()
# client.chat.completions.create()는 채팅 방식의 모델 응답을 생성하는 함수야.

# 역할 :
# 프롬프트(사용자 메시지)를 모델에게 전달하고
# 모델이 생성한 텍스트를 반환함

# 핵심 :
# 예전에는 openai.ChatCompletion.create()였지만
# 새로운 SDK에서는 client.chat.completions.create() 형태로 바뀜

In [ ]:
# 1. zero-shot prompt
# 에시없이 지시사항만 던지는 것 - LLM 의 기본기능
prompt = "이 문장의 감정을 분류해 : '오늘의 점심 메뉴가 품절되어서 너무 슬퍼.'"
print(ask_llm(prompt,temp=1))

In [ ]:
# 2. few-show Prompting
# 이렇게 하는 거야 라고 예시(shot)를 몇개 보여줘서 성능을 높이는 기술
prompt = """
단어를 이모지로 바꿔줘
사과 -> 🍎
자동차 -> 🚗
고양이 -> 🐱
비행기->
"""
print(ask_llm(prompt,temp=1))

In [ ]:
# 3. Chain-of-Thought Prompting Cot(생각의 사슬)
prompt = """
질문 : 5개의 사과중 2개를 먹고 3개를 더 샀어.
사과의 단가는 100원이고
총 지출금액은 800원
총 남은 사과의 개수와 이사람이 사과를 구입해서 사용한 비용을 예상해줘
"""
print(ask_llm(prompt,temp=1))

In [ ]:
# 4. self-Consistnecy(자기일관성) ⭐
    # 원하는 문맥이 나올때까지 반복 돌리기? 그리고 적절한것을 사용자가 선택
    # 혹은 검토용 모델이 선정
# 한번만 묻지 않고 여러번(예 : 3번) 물어본 뒤 가장 많이 나온 답을 채택한다.
question = '철수는 학교까지 10분 걸려, 왕복은 몇분걸릴까?'
answer = []

for _ in range(3):
    answer.append(ask_llm(question))
print(f'수집된 답변들 : ', answer)
from collections import Counter
counter = Counter(answer)
counter

In [ ]:
# 5. Generate Knowlege Prompting(지식생성)
# 바로 답하지말고 관련된 지식을 먼저 생성하 뒤에 그 지식을 바탕으로 답하게 한다.
# 1단계 : 지식생성
knowledge = ask_llm('골프라는 스포츠에 대해 사실적인 지식 3가지만 나열해줘')
# 2단계 : 지식을 활용해 답변
prompt = f"""
다음 지식을 참고해서 '골프에서 홀인원이 왜 어려운지' 설명해줘.
[지식] : {knowledge}
모든 답변은 한글 혹은 영어로 작성
"""
print(ask_llm(prompt))

In [ ]:
# 6. Prompt Chaining(프롬프트 체이닝)
# 복잡한 일을 한번에 시키지 않고 A작업의 결과를 B작업의 입력으로
# 순차적으로 넘겨주는 파이프라인
# Step 1 : 주제 추출
text = '이메일 : 안녕하세요, 이번주 금요일 회의는 2시로 변경되었습니다.'
topic = ask_llm(f'다음 텍스트에서 핵심 주제만 단어로 뽑아줘 출력은 한글로 : {text}')

# Step 2 : 답장 작성
reply = ask_llm(f"'{topic}'에 대해 '알겠습니다'라는 정중한 답장 메일을 써줘")
print(reply)

In [ ]:
# 7. Retrieval Augmented Generation(RAG 검색 증강 생성)
# 이론 : LLM 이 모르는 외부데이터(회사문서등)을 찾아서 (Retrieval)프롬프트에 넣어주고 답하게 한다.

# 가상의 검색된 문서
retrieved_doc = "문서내용 : 우리회사의 재책 근무는 매주 수요일 가능하다."

prompt = f'''
아래[참조문서]를 기반으로 답변해, 문서에 없으며 모른다고 해.
[참조문서] : {retrieved_doc}
질문 : 재택근부는 언제 할 수 있어?
'''

print(ask_llm(prompt))

In [ ]:
# 8. Automaric Reasoning and Tool-use 자동 추론 및 도구 사용
# LLM이 스스로 계산기나 검색엔진 같은 도구가 필요한지 판단하고 호출형식을 뱉어내는 것
prompt = '''
계산이 필요하면  [CALC: 수식] 이라고 출력해.
질문  : 3452 * 192 는 뭐야?
'''
# 실제 내부적으로 파이썬 코드로 계산한다.
response = ask_llm(prompt)
print(response)

In [ ]:
prompt = """
너는 자동 도구 선택 시스템이야.
다음과 같은 도구를 사용할 수 있어.

1. 계산기 -> [CALC:수식]
2. 날씨 조회 -> [WEATHER:도시명]
3. 일반질문 ->  직접입력

규칙 : 
계산이 필요하면  [CALC:...] 출력
날씨정보가 필요하면 [WEATHER:도시명]
그외는 일반적인 답변

질문 :
서울의 내일 날씨는 어때?
"""

import re
def process_response(text):
    # 계산기
    calc = re.findall(r'\[CALC:\s*(.*?)\]',text)
    if calc:
        expr = calc[0]
        return eval(expr)
    # 날씨
    weather = re.findall(r'\[WEATHER:\s*(.*?)\]',text)
    if weather:
        city = weather[0]
        return 'NotImplementedError'
    
response = ask_llm(prompt)
print(response)

for res in response.split('\n'):
    process_response(res)
##

[WEATHER:서울]


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ['OPEN_WEATHER_KEY']
lat = 37.25
lon = 126.45
url = f'https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_KEY}'
import requests
response = requests.get(url)
response.text
##

'{"coord":{"lon":126.45,"lat":37.25},"weather":[{"id":800,"main":"Clear","description":"clear sky","icon":"01d"}],"base":"stations","main":{"temp":287.63,"feels_like":286.99,"temp_min":287.63,"temp_max":287.63,"pressure":1016,"humidity":71,"sea_level":1016,"grnd_level":1016},"visibility":10000,"wind":{"speed":7.45,"deg":237,"gust":8.35},"clouds":{"all":0},"dt":1763955521,"sys":{"type":1,"id":8093,"country":"KR","sunrise":1763936548,"sunset":1763972371},"timezone":32400,"id":1844106,"name":"Daebudo","cod":200}'

In [ ]:
# 9. Automatic Prompt Engineer (APE)
# 사람이 프롬프트는 짜는 게 아니라 LLM에게 좋은 프로프트를 써줘 라고 시키는 것
task = '고객의 리뷰에서 감정을 분석하는 작업'
prompt = f'''
나는  이{task}을 하려고해.
이작업을 수행하기에 가장 환벽한 프롬프트 지시문을 간결하게 작성해줘
'''

best_prompt = ask_llm(prompt)
print(best_prompt)

다음 고객 리뷰를 읽고 감정을 분석해 JSON 형식으로 응답하시오.

- overall_sentiment: "positive" | "negative" | "neutral" 중 하나
- intensity: 0 ~ 5의 정수(감정의 강도)
- key_factors: 감정의 주요 원인으로 꼽을 수 있는 최대 3개의 키워드 배열
- reason: 감정의 근거를 한 문장으로 간략 요약
- improvement: 개선 포인트가 있으면 한 문장으로 제안(없으면 비워도 됨)

출력 예시:
{
  "overall_sentiment": "positive",
  "intensity": 4,
  "key_factors": ["가격", "품질"],
  "reason": "가성비가 좋고 품질에 만족했기 때문입니다.",
  "improvement": "배송 시간을 더 단축하면 좋겠습니다."
}


In [37]:
# 11. Active-Prompt
# LLM이 답변하기 애매하거나 불확실한 문제를 찾아내서 사람에게 이것좀 가르쳐 주세요(예시추가) 라고 요청하는 방식
# LLM에게 문제를 풀게하고 '확신도(Confidence score)를 묻는다 낮으면 그 문제를 few-shot에 예제로 추가

# 프롬프트 생성
import json
def build_prompt(task:str, examples:list, query:str) -> str:
    prompt = f'작업: {task}\n\n'
    for ex in examples:
        example_json = json.dumps({'answer': ex['answer'], 'confidence': ex.get('confidence', 0.9)})
        prompt += f"예시 입력 : {ex['input']}\n예시출력:{example_json}\n\n"
    prompt += f'입력:{query}\njson 형식으로 answer ,confidence 반환'
    return prompt

def call_llm(prompt:str, model='gpt-5-nano') -> dict:
    resp = client.chat.completions.create(
        model=model,
        messages = [{'role':'user', 'content':prompt}]
    )
    text = resp.choices[0].message.content
    try:
        return json.loads(text)
    except:
        return {'answer':text, 'confidence':0.0}
# Active-prompt 루프
def active_prompt(task:str, query:str, examples=None):
    if examples is None:
        examples = []
    for i in range(4):
        prompt = build_prompt(task, examples, query)
        result = call_llm(prompt)
        print(f"\nIteration : {i+1}: answer={result['answer']}, \
               confidence={result.get('confidence',0)}")
        if result.get('confidence',0) >= 0.75:
            break
        else:
            # 낮은 confidence -> few shot 예제로 추가
            examples.append({"input":query,'answer':result['answer'], 
                             "confidence":result.get('confidence',0)})
    return result
# 실행 : 명확한 문제
task1= '숫자가 소수인지 판단'
query1 = '97은 소수인가요?'
res1 = active_prompt(task1,query1)
print(res1)
# 실행 : 애매한 문제


Iteration : 1: answer=97은 소수입니다.,                confidence=0.99
{'answer': '97은 소수입니다.', 'confidence': 0.99}


In [ ]:
# 12. Directional Sti
##

In [ ]:
# 13. Program-Aided Language Models(PAL)
# 수학문제나 날짜 계산을 말로 풀지말고 파이썬 코드를 짜서 해결하도록 유도
# LLM 은 코딩을 더 잘한다. <-누구보다..?
prompt = '''
질문 : 2025년 11월 24일에서 90일 후는 무슨요일이야?
이 문제를 해결하는 python코드를 작성하고 해당 코드를 이용해서 알려줘
'''
print(ask_llm(prompt))

다음은 파이썬으로 2025-11-24에서 90일 후의 요일을 구하는 코드와 그 결과입니다.

코드:
```python
from datetime import date, timedelta

start = date(2025, 11, 24)
target = start + timedelta(days=90)

print(target)              # 2026-02-22
print(target.strftime('%A'))  # Wednesday
```

결과:
- 날짜: 2026-02-22
- 요일: Wednesday

따라서 2025년 11월 24일에서 90일 후의 요일은 수요일입니다.


In [ ]:
# 14. ReAct(Reason -> Act) ⭐
# 생각(Reason)하고 -> 행동(Act)하고 -> 관찰(Observation)하는 과정을 반복하면서 문제를 해결
# 에이전트(Agent)의 기초
prompt = '''
질문 : 손흥민 나이에 10살을 더하면?
다음 형식에 따라서 진행
Thought: 손흥민의 생년월일을 검색한다
Action : Search[손흥민의 생일]
Observation : (검색결과 기디림)
'''
print(ask_llm(prompt))

In [41]:
# 15. Reflexion(리플렉션 / 반성)
# 모델이 틀린답을 냈을 때 왜 틀렸는지 반성(Reflection)하고 다시 답을 출력하는 과정
wrong_answer = "파이썬 리스트 추가하수는 push()입니다."
prompt = f'''
이전 답변 : {wrong_answer}
이답변은 틀렸어. 파이썬 문법에 맞지 않거든.
오류 원인을 분석(Reflection)하고 올바른 답을 수정해서 알려줘.'''
print(ask_llm(prompt))

죄송합니다. 이전에 파이썬 리스트의 추가 방법으로 push()를 언급한 것은 잘못되었습니다. 이는 언어 간 API 차이를 혼동한 잘못된 주장입니다. 원인을 반성해 보면:

- 문제의 본질은 언어별 리스트(또는 배열) API를 혼동한 점에 있습니다. JavaScript의 배열이나 다른 언어에서 push가 일반적이지만, 파이썬의 리스트에는 push가 없습니다.
- 파이썬에서 리스트에 원소를 추가하는 표준 방법은 append()와 extend()이며, 필요에 따라 insert()로 특정 위치에 삽입합니다.
- push()를 시도하면 AttributeError가 발생합니다: 'list' object has no attribute 'push'.

올바른 내용과 사용법은 다음과 같습니다.

필수적으로 알아둘 점
- append(x): 리스트의 맨 끝에 하나의 원소 x를 추가합니다.
- extend(iterable): iterable의 모든 원소를 리스트의 끝에 추가합니다.
- insert(index, x): index 위치에 x를 삽입합니다. (그 자리에 원래 있던 원소와 뒤의 원소들이 밀려납니다.)
- 스택처럼 쓰려면 append로 원소를 push하고, pop으로 꺼냅니다.
- 큐처럼 쓰려면 collections.deque를 사용하면 left에서 popleft로 꺼내는 방식도 있습니다.

예시 코드
- 단일 원소 추가
  lst = [1, 2, 3]
  lst.append(4)  # [1, 2, 3, 4]

- 다수 원소 추가
  lst.extend([5, 6])  # [1, 2, 3, 4, 5, 6]

- 특정 위치에 삽입
  lst.insert(2, 'a')  # [1, 2, 'a', 3, 4, 5, 6]

- 스택 스타일 사용
  stack = []
  stack.append('x')
  stack.append('y')
  top = stack.pop()  # 'y'

- 큐 스타일 사용 (효율적으로)
  from collections import deque
  

In [ ]:
# 16. Multimodel Cot
# 텍스트뿐만 아니라 이미지를 함꼐 보면서 단계별로 추론하는 것
client = OpenAI()

# 로컬파일은 직접 지정 X
# uploaded = client.files.create(
#     file = open(r''),
#     purpose = 'vision'
# )
# file_id = uploaded.id 

response = client.chat.completions.create(
    model='gpt-5-nano',
    messages=[
        {
            "role": "user",
            "content": (
                "다음 이미지를 보면서 상황을 단계별로 추론해서 설명해줘.\n\n"
                "이미지 URL: https://images.ctfassets.net/4cd45et68cgf/7KlXzPyiOBZQiGSBDlYJbe/"
                "0a64bd739f09271ade61c8640a1a2179/SquidGame3_SpecialTeaser_.jpg?w=2000\n\n"
                "요청 형식: 단계별 추론(Chain-of-Thought)을 보여주고, 최종 결론을 간단히 요약해줘."
            )
        }
    ]
)

# 모델 응답 출력
print(response.choices[0].message.content)


죄송하지만 요청하신 "단계별 추론(Chain-of-Thought)" 형식으로 내부 사고 과정을 자세히 보여드릴 수는 없어요. 대신 이미지를 바탕으로 간단한 관찰 요약과 핵심 해석, 최종 결론을 간략히 정리해 드리겠습니다.

참고: 이미지를 직접 확인하지 못하므로 구체적 디테일은 다를 수 있습니다. 이미지 내용을 간단히 텍스트로 요약해 주시면 더 정밀하게 분석해 드릴 수 있습니다.

실제 이미지에서 보일 법한 요소들에 기반한 간단한 해석 예시
- 분위기와 톤: 어두운 색채와 강한 대비로 긴장감과 위험의 분위기를 전달하는 유형의 시각적 구성일 가능성이 큽니다.
- 의상과 마스크: 초록색 점프수트나 작업복과 검은색 경비/관리자복, 그리고 마스크(특정 도형이 보이는 경우)가 권력 구조와 계급 구분을 상징하는 전형적인 요소일 수 있습니다.
- 구성 의도: 참가자 대 경비/관리진의 대립을 암시하거나, 감시와 규칙의 강제성을 부각시키려는 연출일 가능성이 큽니다.
- 텍스트/타이틀: 파일명에 "SquidGame3_SpecialTeaser"가 보인다면 이 영상이 시리즈의 3편(특별 예고편)임을 암시하는 경우가 많습니다.
- 해석 가능한 시나리오
  - 시즌 3의 예고편으로서 새로운 게임 포맷이나 규칙의 도입 가능성을 암시한다.
  - 기존의 권력 관계와 감시 체제가 다시 강조되며, 미스터리한 요소나 새로운 참가자/참가자들 간의 긴장이 예고된다.
  - 팬들에게 앞으로의 갈등 구조와 서사의 확장을 암시하는 분위기다.
- 최종 결론 요약
  - 이 teaser는 Squid Game 3의 스페셜 예고편일 가능성이 높으며, 새로운 게임의 도입과 권력 관계의 재강화를 암시하는 분위기를 전달한다.

추가로 더 구체적으로 분석하려면 이미지의 핵심 요소를 간단히 텍스트로 묘사해 주시거나 이미지를 업로드해 주세요. 그러면 그에 맞춰 보다 정확한 관찰과 해석, 그리고 간결한 결론을 드리겠습니다.


In [44]:
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "이 이미지를 보고 상황을 설명해줘."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://images.ctfassets.net/4cd45et68cgf/7KlXzPyiOBZQiGSBDlYJbe/0a64bd739f09271ade61c8640a1a2179/SquidGame3_SpecialTeaser_.jpg?w=2000"
                    }
                }
            ]
        }
    ]
)

print(response.choices[0].message.content)


다음 포스터를 바탕으로 상황을 설명하면 이렇습니다.

- 전체 분위기: 해가 지는 석양 배경에 달이 떠 있는 분위기로, 다소 신비롭고 긴장감 있는 톤입니다. 두 인물이 화면 앞쪽에 크게 배치되어 시선을 강하게 끕니다.
- 주인공들: 왼쪽에 핑크색 상의를 입은 소녀, 오른쪽에 청록색 상의를 입은 소년이 각각 반쪽 얼굴을 정면으로 바라보고 있습니다. 두 캐릭터 모두 만화적이고 귀여운 CG 스타일로 그려져 있습니다.
- 의상/소품: 소녀는 보라색 머리핀과 똘똘이 같은 포니테일이 특징이고, 소년은 모자 같은 아이템을 쓴 듯 보입니다.
- 텍스트와 정보: 하단에 큰 한국어 글자 “오징어게임 3”이 눈에 띄게 배치되어 있습니다. 그 아래로 “ONLY ON NETFLIX | 2025년 공개”라는 문구가 있어, 이 작품이 넷플릭스 독점으로 2025년에 공개될 예정임을 알립니다.
- 상황적 해석: 포스터의 구성은 두 주인공이 중요한 역할을 맡은 모험이나 미스터리 중심의 이야기를 암시합니다. 제목이 시사하듯이 기존의 “오징어게임” 세계관과 연결되거나 확장된 이야기일 가능성을 내포합니다. 다만 이는 포스터에 의한 예시적 분위기 표현일 뿐, 구체적인 줄거리나 설정은 공식 발표를 기다려야 확인할 수 있습니다.

요약하면: 넷플릭스의 2025년 공개 예정인 “오징어게임 3”의 공식 포스터 같으며, 해가 지는 배경 속에 두 어린 캐릭터가 앞면으로 맞서는 모습으로 모험과 긴장을 암시하는 분위기를 전달합니다.


✔ 반드시 지켜야 하는 포인트
1) image_url의 값은 딕셔너리 형태여야 함
{"image_url": {"url": "이미지주소"}}

2) 전체 메시지 content는 리스트여야 함

문자(text)와 이미지(image_url)를 섞어서 넣기 위함

3) client.chat.completions.creates → 오타

정확히는

client.chat.completions.create

4) 모델이 멀티모달 지원 모델이어야 함

gpt-5-nano는 텍스트만 지원할 수도 있음 → 그 경우 무시됨
(이미지를 텍스트로 인식 못 했기 때문에 방어적 답변이 나온 것)

추천: 멀티모달 확실히 지원되는 모델

예:

gpt-4o-mini

gpt-4o

gpt-5.1 (multimodal 기능 포함)

In [45]:
# 17. Graph Prompting (그래프 프롬프팅
# 데이터를 텍스트가 아니라 그래프(노드와 연결과계)형태로 설명하여 관계 추론 능력을 높인다.
prompt = '''
- (철수) -- 친구 -- (영희)
- (영희) -- 존경 -- (선생님)
- (선생님) -- 제자 -- (철수)

질문 : 철수화 선생님의 관계를 그래프구조를 보고 설명해줘
'''
print(ask_llm(prompt))


그래프 구조로 보면 다음과 같습니다.

- 노드: 철수, 영희, 선생님
- 간선(관계 라벨):
  - 철수 -- 친구 -- 영희
  - 영희 -- 존경 -- 선생님
  - 선생님 -- 제자 -- 철수

설명
- 철수와 선생님의 직접적인 관계는 제자-스승 관계입니다. 즉, 철수는 선생님의 제자이다.
- 영희와 선생님의 관계는 영희가 선생님을 존경한다는 것이라는 간접 관계가 있습니다.
- 철수와 영희는 친구이고, 영희가 선생님을 존경한다는 사실이 있을 뿐이므로, 철수는 영희를 통해 간접적으로 선생님과도 연결될 수 있습니다.

그래프의 모양은 세 노드로 이루어진 삼각관계(triangle)처럼 보이며, 각 간선에 관계 라벨이 붙어 있습니다. 필요하시면 이 그래프를 코드나 다이어그램으로 더 자세히 표현해 드릴게요.


In [47]:
# 문서를 그래프구조 바꿔서 모델에 입력
# 문서 -> 지식 그래프 -> 그래프 프롬프트  -> LLM -> RAG
prompt = '''

HR 부서 -- 관리 --> 채용 프로세스
채용 프로세스 -- 포함 --> 이력서 검토
이력서 검토 --  검토자 --> 답장
이력서 검토 -- 검토자 --> HR매니저

사용자 질문 : "채용과정이 HR부서는 어떤 역할을 하는지?"

지식그래프 기반으로 답변해줘
'''
print(ask_llm(prompt))

다음은 주어진 지식그래프를 바탕으로 한, 채용 과정에서 HR부서의 역할을 지식그래프 관점으로 정리한 내용입니다.

그래프 구조 요약 (주요 노드와 관계)
- 노드: HR 부서, 채용 프로세스, 이력서 검토, 답장, HR매니저
- 관계(에지):
  - HR 부서 -- 관리 --> 채용 프로세스
  - 채용 프로세스 -- 포함 --> 이력서 검토
  - 이력서 검토 -- 검토자 --> 답장
  - 이력서 검토 -- 검토자 --> HR매니저

각 노드의 역할(지식그래프 관점으로 해석)
- HR 부서
  - 역할: 전체 채용 운영을 관리하는 주체
  - 그래프상에서는 채용 프로세스를 관리하는 최상위 노드로 표현되며, 정책 수립, 표준화, 준수 여부 점검, 후보자 경험 관리 등을 포괄적으로 책임.
- 채용 프로세스
  - 역할: 이력서 검토를 포함하는 채용의 구체적 절차
  - 그래프상에서 이력서 검토를 포함하는 구성 요소로 연결되어 있어, 채용의 흐름과 단계가 어떻게 구성되는지 표현.
- 이력서 검토
  - 역할: 채용의 초기 검증 단계로서 이력서를 평가하고 적합 여부를 판단하는 작업
  - 그래프상에서 채용프로세스의 핵심 하위 단계이며, 두 명의 검토자와 연결되어 검토가 이뤄짐을 표현.
- 답장
  - 역할: 이력서 검토 단계의 검토자 중 하나로, 후보자와의 커뮤니케이션(피드백 제공, 합격/참가 여부 알림 등)을 담당
  - 그래프상에서 이력서 검토의 검토자 역할로 배치되어 후보자와의 응답/소통과 관련된 업무를 담당하는 주체로 해석.
- HR매니저
  - 역할: 이력서 검토의 또 다른 검토자이자, 최종 의사결정에 관여하는 주체
  - 그래프상에서 채용결정의 책임자이거나 최종 승인권한을 가진 역할로 해석할 수 있음.

실무적 시사점
- HR 부서는 채용 프로세스를 관리하는 주체이므로, 초기 이력서 검토의 품질과 속도에 직접적인 영향력을 행사합니다.
- 이력서 검토 단계에서 두 유형의 검토자(답장/HR매니저)가 존재하는 구조는 후보자 커뮤니케이션과 최종 의사결정이 분리

In [48]:
# 18. Meta-prompting(메타 프롬프팅)
# 하나의 거대한 문제르 해결하기위해서 llm 스스로 하위 프롬포트를 여러개 생성하고
# 관리하는 Meta(최상의) 기법
# 오케스트라 지휘자

# 문제 : 중소기업이 AI 도입 전략을 세울때 핵심단계를 알려줘
# LLM 판단 
    # 전략 전문가
    # 기술 전문가
    # 예산 : ROI 전문가
# 3명이 필요하다고 하면 스스로 하위 프롬프트 생성  --> 각각 실행 -> 종합

meta_prompt = '''
우리는 다음 문제를 해결하려고 한다
중소기업이 AI 도입 전략을 세울때 핵심단계를 알려줘
 
이 문제를 해결하기 위해 필요한 전문가 3명을 정의하고
각 전문가에 줄 개별 프롬프트를 만들어라

출력형식 :
1. 전문가 이름 :
프롬프트 :
2. 전문가2 이름 :
프롬프트 : 
3. 전문가3 이름 :
프롬프트 :
'''
print(f'메타 프롬프트를 생성 ...')
experts  = ask_llm(meta_prompt)
print(experts)

메타 프롬프트를 생성 ...
1. 전문가 이름 : 김하은
프롬프트 :
당신은 중소기업의 AI 도입 전략 및 거버넌스 전문가입니다. 중소기업의 예산 제약, 인력 규모, 산업 특성 등을 고려하여, AI 도입의 핵심 단계와 산출물을 구체적으로 제시하라. 핵심 단계의 순서를 포함하고, 각 단계의 목표, 주요 산출물(문서 템플릿 포함), 책임자/관할 부서, 예상 소요 기간(주 단위), 예산 범위, 성공 지표를 제시하라. 또한 SMB에 특화된 실행 로드맹(6-18개월)과 경영진 보고 프레임, 위험 관리 및 법적/윤리적 고려사항(데이터 프라이버시, 규정 준수)도 포함하라. 산업별 예시(제조/소매/서비스)와 기능별 적용 예시를 함께 제공하되, 표 없이도 바로 실행 가능한 형태로 제시하고, 필요한 경우 간단한 체크리스트와 산출물 예시를 함께 제시하라. 마지막으로 벤더/도입 파트너 선정 시 고려해야 할 기준과 평가 체크리스트도 포함하라.

2. 전문가 이름 : 이민수
프롬프트 :
당신은 데이터 엔지니어링/데이터 거버넌스/ML Ops 전문가입니다. 중소기업의 AI 도입에서 데이터 준비와 파이프라인 설계가 핵심인 만큼, 현재 데이터 현황 진단에서 시작해 데이터 품질 관리, 데이터 카탈로그/메타데이터 관리, 데이터 보안 및 프라이버시, 데이터 저장 전략(데이터 레이크 대 데이터 웨어하우스), 데이터 파이프라인 설계(ETL/ELT), 플랫폼 선택(클라우드 기반 vs 온프렘), 데이터 거버넌스 프레임워크, 모델 운영(MLOps) 프로세스까지 포함하는 실행 계획을 제시하라. SMB 예산에 맞춘 간이 아키텍처 예시와 비용 추정도 포함하고, 벤더 관리/계약 시 유의점, 데이터 품질 규칙, 테스트 전략, 데이터 이용 정책(권한 관리 및 접근 제어) 예시를 제공하라. 또한 Pilot를 위한 데이터 샘플 목록, 품질 지표 대시보드 예시, 개인정보 보호를 위한 기본 원칙과 준수 포인트를 제시하고, 도입 초기의 위험 요소와 대처 방안을 함께 제공하라. 필요 시 산업별 도메인별 맞춤 조언을 포함하되, 

In [ ]:
import re
exp_prompts = re.findall(r'프롬프트 : \s*(.*)',experts)
# 각 전문가  llm 호출
result = [ask_llm(prompt) for  prompt in exp_prompts]

# 결과 종합 프롬프트 
final_prompt = '''
3명의 전문가가 낸 분석 결과
1. {result[0]}
2. {result[1]}
3. {result[2]}

위 내용을 종합해서 중소기업이 AI도입 전략을 세울때 핵심 단계를 정리해서
작성해 단계적으로, 명확하게, 문장이나 문맥에 어색함 없이, 한글로 작성해
'''
print(ask_llm(final_prompt))